In [1]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np


In [2]:
texts = [
    # Positive
    "I love this product.",
    "This movie is amazing.",
    "The food was delicious.",
    "I am very happy today.",
    "The service was excellent.",
    "The noodles were tasty.",
    "The food was good.",
    "The staff was friendly.",
    "The experience was great.",
    "The product worked well.",

    # Negative
    "I hate this product.",
    "This movie is boring.",
    "The food was terrible.",
    "I am very sad today.",
    "The service was bad.",
    "The noodles were not tasty.",
    "The food was not good.",
    "The movie was not amazing.",
    "The service was not excellent.",
    "I am not happy today.",

    # Neutral
    "This product is okay.",
    "The movie was average.",
    "The food was normal.",
    "I feel fine today.",
    "The service was acceptable.",
    "The noodles were okay.",
    "The experience was neither good nor bad.",
    "The product was average.",
    "The staff was available.",
    "The food was decent."
]

# 0 = Negative, 1 = Positive, 2 = Neutral
labels = np.array([
    # Positive
    1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

    # Negative
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

    # Neutral
    2, 2, 2, 2, 2, 2, 2, 2, 2, 2
])

In [3]:
vocab_size = 1000
max_length = 8

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)

X = pad_sequences(
    sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

print("Word Index:")
print(tokenizer.word_index)

print("\nPadded Sequences:")
print(X)

print("\nInput Shape:", X.shape)
print("Labels Shape:", labels.shape)

Word Index:
{'<OOV>': 1, 'the': 2, 'was': 3, 'i': 4, 'food': 5, 'this': 6, 'product': 7, 'not': 8, 'movie': 9, 'today': 10, 'service': 11, 'is': 12, 'am': 13, 'noodles': 14, 'were': 15, 'good': 16, 'amazing': 17, 'very': 18, 'happy': 19, 'excellent': 20, 'tasty': 21, 'staff': 22, 'experience': 23, 'bad': 24, 'okay': 25, 'average': 26, 'love': 27, 'delicious': 28, 'friendly': 29, 'great': 30, 'worked': 31, 'well': 32, 'hate': 33, 'boring': 34, 'terrible': 35, 'sad': 36, 'normal': 37, 'feel': 38, 'fine': 39, 'acceptable': 40, 'neither': 41, 'nor': 42, 'available': 43, 'decent': 44}

Padded Sequences:
[[ 4 27  6  7  0  0  0  0]
 [ 6  9 12 17  0  0  0  0]
 [ 2  5  3 28  0  0  0  0]
 [ 4 13 18 19 10  0  0  0]
 [ 2 11  3 20  0  0  0  0]
 [ 2 14 15 21  0  0  0  0]
 [ 2  5  3 16  0  0  0  0]
 [ 2 22  3 29  0  0  0  0]
 [ 2 23  3 30  0  0  0  0]
 [ 2  7 31 32  0  0  0  0]
 [ 4 33  6  7  0  0  0  0]
 [ 6  9 12 34  0  0  0  0]
 [ 2  5  3 35  0  0  0  0]
 [ 4 13 18 36 10  0  0  0]
 [ 2 11  3 24  0

In [4]:
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, max_length, vocab_size, embed_dim):
        super().__init__()

        # Token Embedding: converts word IDs into vectors
        self.token_embedding = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )

        # Positional Embedding: gives position/order information
        self.position_embedding = layers.Embedding(
            input_dim=max_length,
            output_dim=embed_dim
        )

    def call(self, x):
        # Create position numbers: [0, 1, 2, 3, ...]
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)

        # Convert token IDs into vectors
        token_emb = self.token_embedding(x)

        # Convert positions into vectors
        position_emb = self.position_embedding(positions)

        # Add token meaning + position information
        return token_emb + position_emb

In [5]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()

        # Multi-Head Self-Attention
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        # Feed Forward Network
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])

        # Add & Normalize layers
        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()

    def call(self, inputs):
        # Self-attention: Query, Key, Value all come from same input
        attention_output = self.attention(inputs, inputs)

        # First Add & Normalize
        out1 = self.layernorm1(inputs + attention_output)

        # Feed Forward Network
        ffn_output = self.ffn(out1)

        # Second Add & Normalize
        out2 = self.layernorm2(out1 + ffn_output)

        return out2

In [6]:
embed_dim = 16
num_heads = 2
ff_dim = 32
num_classes = 3

inputs = layers.Input(shape=(max_length,))

# Token Embedding + Positional Embedding
x = TokenAndPositionEmbedding(
    max_length=max_length,
    vocab_size=vocab_size,
    embed_dim=embed_dim
)(inputs)

# Transformer Encoder Block
x = TransformerBlock(
    embed_dim=embed_dim,
    num_heads=num_heads,
    ff_dim=ff_dim
)(x)

# Convert token-level output into sentence-level output
x = layers.GlobalAveragePooling1D()(x)

# Optional Dense layer
x = layers.Dense(32, activation="relu")(x)

# Output Layer for 3 classes
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

In [7]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, 8, 16)          │        16,128 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 8, 16)          │         3,296 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 16)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,067 (78.39 KB)

 Trainable params: 20,067 (78.39 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
epoch = 50
batch_size = 4
model.fit(X,labels,epochs=epoch ,batch_size=batch_size,verbose=1)


Epoch 1/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.3333 - loss: 1.1518
Epoch 2/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.3333 - loss: 1.0916 
Epoch 3/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.3000 - loss: 1.0692 
Epoch 4/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4000 - loss: 1.0622 
Epoch 5/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4667 - loss: 1.0365 
Epoch 6/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5333 - loss: 0.9980 
Epoch 7/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5000 - loss: 0.9798 
Epoch 8/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7667 - loss: 0.9362 
Epoch 9/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7333 - loss: 0.9227 
Epoch 10/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5667 - loss: 0.9098 
Epoch 11/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6667 - loss: 0.8604 
Epoch 12/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7667 - loss: 0.8308 
Ep

In [9]:
class_names = {
    0: "Negative",
    1: "Positive",
    2: "Neutral"
}

test_sentences = [
    "I love the food",
    "The movie was terrible",
    "The service was okay",
    "The product was not good",
    "I feel fine today",
    "The staff was excellent",
    "The experience was average"
]

test_sequences = tokenizer.texts_to_sequences(test_sentences)

test_padded = pad_sequences(
    test_sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

predictions = model.predict(test_padded)

for sentence, prediction in zip(test_sentences, predictions):
    predicted_class = np.argmax(prediction)
    confidence = prediction[predicted_class]

    print("\nSentence:", sentence)
    print("Raw Prediction:", prediction)
    print("Predicted Class:", class_names[predicted_class])
    print("Confidence:", confidence)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step

Sentence: I love the food
Raw Prediction: [2.9745686e-04 9.9684346e-01 2.8590674e-03]
Predicted Class: Positive
Confidence: 0.99684346

Sentence: The movie was terrible
Raw Prediction: [9.9915290e-01 5.0006568e-04 3.4710596e-04]
Predicted Class: Negative
Confidence: 0.9991529

Sentence: The service was okay
Raw Prediction: [0.00254692 0.00204223 0.99541086]
Predicted Class: Neutral
Confidence: 0.99541086

Sentence: The product was not good
Raw Prediction: [9.9902785e-01 6.9734931e-04 2.7485829e-04]
Predicted Class: Negative
Confidence: 0.99902785

Sentence: I feel fine today
Raw Prediction: [0.00206818 0.0023061  0.99562573]
Predicted Class: Neutral
Confidence: 0.99562573

Sentence: The staff was excellent
Raw Prediction: [1.3842648e-04 9.9414331e-01 5.7182596e-03]
Predicted Class: Positive
Confidence: 0.9941433

Sentence: The experience was average
Raw Prediction: [8.3703652e-04 5.6259045e-03 9.9353701e-01]
Predicted Class: Neutral
Confidence: 0